# Evaluate HelpSteer2 Many-Objective Adapter Merges

This notebook evaluates fixed weighted merges of five already trained GPT-2 LoRA adapters for the HelpSteer2 attributes **helpfulness**, **correctness**, **coherence**, **complexity**, and **verbosity**.

It does not train adapters, compute the relationship matrix $R$, or run the M1 preference-to-coefficient mapping. It only checks that many-objective adapter merging works and records generated responses.

## Clone or update the repository

The following cell always starts from `/content`. It updates an existing valid repository or clones a fresh copy. This prevents nested paths such as `/content/master-thesis/master-thesis`.

In [ ]:
from pathlib import Path
import os
import subprocess

repo_path = Path("/content/master-thesis")
repo_url = "https://github.com/NZhang137/master-thesis.git"

os.chdir("/content")

if (repo_path / ".git").is_dir():
    print("Repository found. Pulling the latest changes...")
    subprocess.run(["git", "-C", str(repo_path), "pull"], check=True)
elif repo_path.exists():
    raise RuntimeError(
        f"{repo_path} exists but is not a Git repository. "
        "Rename or remove it, then run this cell again."
    )
else:
    print("Cloning the repository...")
    subprocess.run(["git", "clone", repo_url, str(repo_path)], check=True)

os.chdir(repo_path)
print(f"Current folder: {Path.cwd()}")

## Show the repository structure

Confirm that the repository contains the `scripts/` and `src/` folders before continuing.

In [ ]:
!pwd
!ls
!ls scripts
!ls src

## Install dependencies

Colab sometimes includes an old `torchao` version that is incompatible with recent Transformers releases. This prototype does not require `torchao`, so the cell removes it before installing the required packages.

In [ ]:
!pip uninstall -y torchao
!pip install -q -U transformers datasets peft accelerate safetensors pandas

If Colab reports that packages were already imported, restart the runtime and begin again from the first cell. If a future environment explicitly requires `torchao`, install a compatible current version with `!pip install -q -U "torchao>=0.16.0"`, then restart the runtime.

## Check whether the HelpSteer2 adapters exist

The evaluation needs all five local PEFT adapter folders. The next cell reports which folders are available.

In [ ]:
from pathlib import Path

adapter_paths = [
    Path("adapters/helpsteer2-gpt2-helpfulness-adapter"),
    Path("adapters/helpsteer2-gpt2-correctness-adapter"),
    Path("adapters/helpsteer2-gpt2-coherence-adapter"),
    Path("adapters/helpsteer2-gpt2-complexity-adapter"),
    Path("adapters/helpsteer2-gpt2-verbosity-adapter"),
]

for path in adapter_paths:
    status = "FOUND" if path.is_dir() else "MISSING"
    print(f"{status:7} {path}")

all_adapters_exist = all(path.is_dir() for path in adapter_paths)
print(f"\nAll adapters available: {all_adapters_exist}")

## Upload the adapter backup if needed

Run the upload and unzip cells only when one or more adapter folders are missing. Select `helpsteer2_adapters.zip` from your computer.

**Important:** The ZIP file is only a local backup of generated adapter weights. It must not be committed to GitHub.

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
!if [ -f helpsteer2_adapters.zip ]; then unzip -o helpsteer2_adapters.zip; else echo "No helpsteer2_adapters.zip found, skipping unzip."; fi
!ls adapters || echo "No adapters folder found."

## Check the adapter files

This lightweight check confirms that each folder contains the expected PEFT configuration and adapter weight file.

In [ ]:
!python scripts/check_helpsteer2_adapters.py

## Evaluate the fixed adapter merges

The script loads the five adapters, evaluates a small collection of fixed coefficient vectors, and generates responses for a short prompt set. This may take several minutes because a fresh model is created for each merge candidate.

In [ ]:
!python scripts/evaluate_helpsteer2_adapter_merges.py

## Inspect the generated results

The script writes the small result file `results/helpsteer2_adapter_merge_generations.csv`.

In [ ]:
!ls results
!head -n 10 results/helpsteer2_adapter_merge_generations.csv

In [ ]:
import pandas as pd

results_df = pd.read_csv("results/helpsteer2_adapter_merge_generations.csv")
print(f"Rows: {len(results_df)}")
results_df.head()

## Understanding the output

Each row contains one generated response for one fixed coefficient vector and one prompt. The five `lambda_...` columns correspond to helpfulness, correctness, coherence, complexity, and verbosity. `merge_name` identifies the tested coefficient setting.

This step only tests many-objective LoRA adapter merging. It does not compute a relationship matrix or apply preference-aware coefficient correction.

## Git safety check

It is acceptable to commit the small CSV result file. Do not commit `adapters/`, `helpsteer2_adapters.zip`, `.safetensors`, `.bin`, checkpoints, or other model files.

In [ ]:
!git status